# Train a model with reinforcement learning (GRPO) on one GPU

This notebook trains **Qwen3-1.7B** with **GRPO** (reinforcement learning). For each prompt the model writes several answers, a scoring function grades them, and training pushes the model toward the better ones.

What it learns comes from a **task**: a small Python file with the data, the prompt and the scoring. Two are built in:
- `math` (default): grade-school math word problems.
- `toolcall`: answer a request with the right JSON function calls.

To train on your own task, write a task file (see `tasks/README.md`) and set `TASK` to its path or URL.

It runs on a single 24 GB card (RTX 3090 / 4090) or bigger. The default 100-step run takes 8 to 18 minutes, depending on the card and task.

Everything here calls `train_grpo.py` in this folder. The same script works from a terminal, over SSH, or from an agent:

```bash
python train_grpo.py --help
```

## 1. Settings

Change these, then run the cells in order. Lower `NUM_GENERATIONS` or `MAX_SEQ_LENGTH` if you run out of memory.

In [ ]:
TASK            = "math"                    # "toolcall", or a path / URL to your own task file
MODEL           = "unsloth/Qwen3-1.7B-Base" # or unsloth/Qwen3-4B-Base; Qwen3-8B-Base on 32 GB+ cards
DATASET         = ""                        # math only: "gsm8k" (default) or "dapo-math" (harder)
MAX_STEPS       = 100                       # 20 for a quick test
MAX_SEQ_LENGTH  = 2048
NUM_GENERATIONS = 4
EVAL_SAMPLES    = 200                       # held-out prompts scored before and after
OUTPUT_DIR      = "/workspace/grpo-runs/notebook"
HF_REPO         = ""                        # e.g. "your-name/qwen3-1.7b-grpo" (needs HF_TOKEN)

## 2. Train

Watch the `reward` column in the log: it should climb as the model learns the answer format and gets more answers right. If `frac_reward_zero_std` stays near 1.0, every answer to a prompt gets the same score and the model learns nothing: give your rewards more partial credit.

In [ ]:
!python train_grpo.py \
    --task "{TASK}" --model "{MODEL}" --dataset "{DATASET}" --max-steps {MAX_STEPS} \
    --max-seq-length {MAX_SEQ_LENGTH} --num-generations {NUM_GENERATIONS} \
    --eval-samples {EVAL_SAMPLES} --output-dir "{OUTPUT_DIR}" --hf-repo "{HF_REPO}"

## 3. Results

In [ ]:
import json, matplotlib.pyplot as plt

results = json.load(open(f"{OUTPUT_DIR}/results.json"))
history = json.load(open(f"{OUTPUT_DIR}/log_history.json"))
print(json.dumps({k: v for k, v in results.items() if k != "settings"}, indent=2))

steps   = [h["step"] for h in history if "reward" in h]
rewards = [h["reward"] for h in history if "reward" in h]
plt.figure(figsize=(8, 3))
plt.plot(steps, rewards, alpha=0.4, label="reward")
k = 10
plt.plot(steps[k-1:], [sum(rewards[i-k+1:i+1]) / k for i in range(k-1, len(rewards))], label=f"{k}-step average")
plt.xlabel("step"); plt.ylabel("reward"); plt.legend(); plt.show()

## 4. Try your model

Loads the base model plus the LoRA adapter you just trained, with the run's own task file, and answers one new prompt. For your own task, write `row` in the shape your task's `load()` returns.

In [ ]:
from train_grpo import load_trained, stop_strings  # first: sets UNSLOTH_VLLM_STANDBY before unsloth loads
import json
from vllm import SamplingParams

# Rebuilds the exact base model and task the run used, plus the trained LoRA.
model, tokenizer, lora, task = load_trained(OUTPUT_DIR)

examples = {
    "math": {"question": "A baker makes 12 trays of 18 cookies and sells 150. How many are left?"},
    "toolcall": {
        "question": "What's the weather in Paris right now?",
        "tools": json.dumps([{"name": "get_weather", "description": "Current weather for a city",
                              "parameters": {"city": {"type": "str", "description": "City name"}}}]),
    },
}
row = examples[TASK]

text = tokenizer.apply_chat_template(task.prompt(row), tokenize=False, add_generation_prompt=True)
params = SamplingParams(temperature=0.0, max_tokens=512, stop=stop_strings(task, tokenizer))
out = model.fast_generate([text], sampling_params=params, lora_request=lora)
print(out[0].outputs[0].text)

## Next steps

- Your own task: copy `tasks/toolcall_task.py`, change the data, prompt and scoring, and set `TASK` to the new file. `tasks/README.md` has the details.
- Bigger model: `unsloth/Qwen3-8B-Base` on a 32 GB card (RTX 5090), or add `--load-in-4bit` on 24 GB.
- Save a full model for serving: `--save-merged`, then serve it with Vast's vLLM template.